# BLOQUE 0: Repaso de la Clase 06 — Estadística y Preprocesamiento

Antes de meternos con IA, Machine Learning y Scikit-Learn, vamos a hacer un ejemplito rápido que sirve para repasar 4 conceptos clave de la Clase 06 (estadística y preprocesamiento), porque son el terreno sobre el que se apoya todo lo que viene: para que un modelo de ML "aprenda" de datos (Clase 07), primero hace falta saber **leerlos** con estas mismas herramientas.

Vamos a repasar estos 4 conceptos sobre el mismo dataset de natalidad que usamos en la Clase 06, para que el repaso quede conectado con algo real y no sea solo teoría suelta.

### 1) Limpieza e Integración
Ningún dataset real llega listo para analizar. **Limpiar** significa decidir qué hacer con nulos (imputarlos con la media/mediana si son numéricos, con la moda o una etiqueta de negocio si son categóricos) y con duplicados (eliminarlos). La clave es no borrar a ciegas: un nulo puede tener una causa de negocio válida detrás (por ejemplo, un campo que simplemente no aplica a ese registro) y no ser necesariamente un error de carga. **Integrar** significa combinar o derivar columnas nuevas a partir de las existentes, para que la información cruda se vuelva accionable — por ejemplo, transformar un valor numérico en una etiqueta de negocio interpretable.

In [ ]:
import pandas as pd
import numpy as np

# =====================================================================
# REPASO 1: Limpieza e Integración
# =====================================================================
print("=== REPASO 1: LIMPIEZA E INTEGRACIÓN ===")

df_raw = pd.read_csv('tasa-natalidad-deis-2000-2024.csv')
print(f"Filas: {df_raw.shape[0]} (años, de 2000 a 2024) | Columnas: {df_raw.shape[1]} (1 índice de tiempo + 25 provincias)")

nulos = df_raw.isnull().sum().sum()
duplicados = df_raw.duplicated().sum()
print(f"Valores nulos totales: {nulos}")
print(f"Filas duplicadas: {duplicados}")

if nulos == 0 and duplicados == 0:
    print("👉 Este dataset del DEIS ya llega limpio (0 nulos, 0 duplicados).")
    print("   Si hubiera nulos, se imputan con media/mediana (numéricas) o moda/etiqueta de negocio (categóricas);")
    print("   si hubiera duplicados, se eliminan con .drop_duplicates(). El procedimiento es siempre el mismo.")

# Integración: creamos una variable derivada de negocio a partir de un valor numérico
natalidad_2024 = df_raw[df_raw['indice_tiempo'] == '01-01-2024'].drop(columns='indice_tiempo').T
natalidad_2024.columns = ['natalidad_2024']
natalidad_2024['categoria_natalidad'] = pd.cut(
    natalidad_2024['natalidad_2024'],
    bins=[0, 8.4, 9.7, np.inf],
    labels=['Baja', 'Media', 'Alta']
)
print("\nVariable derivada 'categoria_natalidad' (Integración) sobre el último año disponible:")
print(natalidad_2024['categoria_natalidad'].value_counts())

### 2) Medidas de Tendencia Central y Dispersión
La **tendencia central** responde "¿dónde está el centro de los datos?": la **media** (promedio) es sensible a valores extremos, la **mediana** (valor que deja 50% de los datos a cada lado) no lo es, y la **moda** es el valor más frecuente. Cuando media y mediana difieren mucho, es señal de asimetría o de outliers. La **dispersión** responde "¿qué tan esparcidos están?": el **desvío estándar** mide la variación promedio respecto a la media, y el **IQR** (rango intercuartílico, Q3 − Q1) mide el ancho del 50% central de los datos, siendo más robusto frente a outliers.

In [ ]:
# =====================================================================
# REPASO 2: Medidas de Tendencia Central y Dispersión
# =====================================================================
print("=== REPASO 2: TENDENCIA CENTRAL Y DISPERSIÓN ===")

serie_nacional = df_raw['natalidad_argentina']

media = serie_nacional.mean()
mediana = serie_nacional.median()
std = serie_nacional.std()
q1 = serie_nacional.quantile(0.25)
q3 = serie_nacional.quantile(0.75)
iqr = q3 - q1

print(f"Media (natalidad nacional 2000-2024): {media:.2f}")
print(f"Mediana: {mediana:.2f}")
print(f"Desvío estándar: {std:.2f}")
print(f"IQR (Q3 - Q1): {iqr:.2f} (entre {q1:.2f} y {q3:.2f})")

print("\n👉 DEDUCCIÓN: la mediana (17.9) es más alta que la media (16.35).")
print("   Esto no se debe a outliers, sino a que la natalidad viene en caída sostenida:")
print("   hay más años 'altos' (principio de la serie) que 'bajos' (final de la serie),")
print("   por eso el valor central pesa distinto que el promedio. Un IQR de ~3.1 puntos")
print("   confirma que, aun con esa caída, la dispersión año a año es moderada.")

### 3) Distribuciones y Correlación
La **distribución** es la forma que toman los datos al graficarlos (histograma): puede ser simétrica (tipo campana/Normal) o sesgada a la izquierda/derecha. La **correlación** (coeficiente de Pearson, entre -1 y 1) mide qué tan asociadas linealmente están dos variables numéricas: cerca de 1 indica que suben juntas, cerca de -1 que una sube cuando la otra baja, y cerca de 0 que no hay relación lineal. Recordar siempre: **correlación no implica causalidad**.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# =====================================================================
# REPASO 3: Distribuciones y Correlación
# =====================================================================
print("=== REPASO 3: DISTRIBUCIONES Y CORRELACIÓN ===")

plt.figure(figsize=(12, 4))

# Distribución: forma de la serie nacional a lo largo de los 25 años
plt.subplot(1, 2, 1)
sns.histplot(serie_nacional, kde=True, bins=10, color='teal')
plt.axvline(media, color='red', linestyle='--', label=f'Media: {media:.1f}')
plt.axvline(mediana, color='green', linestyle='--', label=f'Mediana: {mediana:.1f}')
plt.title('Distribución de la natalidad nacional (2000-2024)')
plt.xlabel('Tasa de natalidad')
plt.legend()

# Correlación: ¿se mueven juntas las provincias grandes?
plt.subplot(1, 2, 2)
provincias_comparar = df_raw[['natalidad_buenos_aires', 'natalidad_cordoba', 'natalidad_santa_fe']]
matriz_corr = provincias_comparar.corr()
sns.heatmap(matriz_corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title('Correlación entre provincias')

plt.tight_layout()
plt.show()

skew = serie_nacional.skew()
corr_ba_cba = matriz_corr.loc['natalidad_buenos_aires', 'natalidad_cordoba']
print(f"Asimetría (skew) de la serie nacional: {skew:.2f}")
print(f"Correlación Buenos Aires vs. Córdoba: {corr_ba_cba:.2f}")
print("\n👉 DEDUCCIÓN: la correlación es altísima (>0.95) porque ambas provincias comparten la misma")
print("   tendencia demográfica del país (la caída de la natalidad es un fenómeno nacional, no local).")
print("   Ojo: esto es correlación, no causalidad — Córdoba no baja su natalidad *porque* baja Buenos Aires.")

### 4) Transformación y Reducción de Dimensionalidad
Para que un algoritmo matemático procese los datos hace falta **transformarlos**: convertir texto a números (`LabelEncoder`, one-hot encoding) y llevar las variables numéricas a una escala comparable (`StandardScaler`), ya que los modelos basados en distancias (como K-Means, que usaremos hoy) son sensibles a la magnitud de cada columna. Cuando hay muchas columnas, **PCA** (Análisis de Componentes Principales) permite comprimirlas en unas pocas dimensiones que conservan la mayor parte de la variabilidad original, simplificando el análisis sin perder información relevante.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# =====================================================================
# REPASO 4: Transformación y Reducción de Dimensionalidad
# =====================================================================
print("=== REPASO 4: TRANSFORMACIÓN Y REDUCCIÓN ===")

# Transformación: escalado de un par de columnas para que queden comparables
columnas_ejemplo = df_raw[['natalidad_buenos_aires', 'natalidad_cordoba']]
scaler_demo = StandardScaler()
columnas_escaladas = scaler_demo.fit_transform(columnas_ejemplo)

print("Antes de escalar (media original):")
print(columnas_ejemplo.mean().round(2).to_dict())
print("Después de escalar con StandardScaler (media ≈ 0, desvío ≈ 1):")
print(pd.DataFrame(columnas_escaladas, columns=columnas_ejemplo.columns).describe().loc[['mean', 'std']].round(2))

# Reducción: PCA sobre todas las provincias (25 años = 25 features por provincia)
provincias_T = df_raw.drop(columns='indice_tiempo').T
X_scaled_demo = StandardScaler().fit_transform(provincias_T)
pca_demo = PCA(n_components=2)
pca_demo.fit_transform(X_scaled_demo)
varianza_total = pca_demo.explained_variance_ratio_.sum() * 100

print(f"\nVarianza explicada al comprimir 25 años en 2 componentes principales: {varianza_total:.1f}%")
print("👉 DEDUCCIÓN: escalar es obligatorio antes de PCA o K-Means porque estos algoritmos miden distancias;")
print("   sin escalar, una columna con números más grandes 'pesaría' más en el resultado solo por su magnitud,")
print("   no porque sea más importante. Este mismo criterio (dividir train/test, ajustar el escalador solo con train)")
print("   lo vamos a retomar formalmente en la Clase 07, con train_test_split de Scikit-Learn.")